In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.profiler import profile, record_function, ProfilerActivity
from transformers import Trainer, TrainingArguments, TrainerCallback

# --- Dataset Definition ---
class CIFAR10Dataset(torch.utils.data.Dataset):
    def __init__(self, split: str, transform):
        self.data = torchvision.datasets.CIFAR10(
            root='./data',
            train=(split == 'train'),
            download=True,
            transform=transform
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx: int):
        image, label = self.data[idx]
        return {"pixel_values": image, "labels": label}

# --- Transforms ---
transform = transforms.Compose([
    transforms.Resize((86, 86)),  # Resize CIFAR-10 images for ResNet
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = CIFAR10Dataset(split="train", transform=transform)
eval_dataset = CIFAR10Dataset(split="test", transform=transform)

# --- Wrapped ResNet101 Model ---
class ResNet101Wrapper(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.resnet = torchvision.models.resnet101(pretrained=False)
        # Replace the final layer with a classifier for CIFAR-10.
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)
        # Inform the Trainer which keys to pass.
        self.model_input_names = ["pixel_values", "labels"]

    def forward(self, pixel_values, labels=None):
        logits = self.resnet(pixel_values)
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
            return {"loss": loss, "logits": logits}
        else:
            return logits

model = ResNet101Wrapper()

# --- Training Arguments ---
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=64,
    logging_dir="./logs",
    logging_steps=50,
    save_strategy="epoch",
    report_to="tensorboard",
    no_cuda=True
)

# --- Profiler Callback ---
class TorchMemoryProfilerCallback(TrainerCallback):
    def __init__(self):
        activities = [ProfilerActivity.CPU]
        # if torch.cuda.is_available():
        #     activities.append(ProfilerActivity.CUDA)
        self.profiler = profile(
            activities=activities,
            schedule=torch.profiler.schedule(
                wait=1, warmup=1, active=5, repeat=1
            ),
            on_trace_ready=torch.profiler.tensorboard_trace_handler("./log/memory"),
            record_shapes=True,
            profile_memory=True,
            with_stack=True,
            with_flops=False,
            with_modules=True,
        )

    def on_train_begin(self, args, state, control, **kwargs):
        print("Starting profiler...")
        self.profiler.start()

    def on_epoch_begin(self, args, state, control, **kwargs):
        print("Starting profiler...")
        self.profiler.step()


    def on_step_end(self, args, state, control, **kwargs):
        self.profiler.step()

    def on_train_end(self, args, state, control, **kwargs):
        print("Stopping profiler...")
        self.profiler.stop()


from transformers import TrainerCallback, TrainerControl, TrainerState

class StepBasedStopCallback(TrainerCallback):
    def __init__(self, stop_after_steps: int):
        self.stop_after_steps = stop_after_steps

    def on_step_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        # Check if the number of steps has reached the limit
        if state.global_step >= self.stop_after_steps:
            print(f"Reached {state.global_step} steps. Stopping training now.")
            control.should_training_stop = True  # Signal to stop training
        return control


# --- Instantiate the Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    callbacks=[TorchMemoryProfilerCallback(), StepBasedStopCallback(2)],
)

# --- Start Training ---
trainer.train()


/home/glaswigian/miniconda3/envs/Huggingface/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/glaswigian/miniconda3/envs/Huggingface/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/glaswigian/miniconda3/envs/Huggingface/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/home/glaswigian/miniconda3/envs/Huggingface/lib/python3.12/site-packages/transformers/training_args.py:1590: FutureWarning: using `no_cuda` is 

Starting profiler...
Starting profiler...


Step,Training Loss


[W205 12:12:08.732157537 CPUAllocator.cpp:245] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event


Reached 2 steps. Stopping training now.
Stopping profiler...


TypeError: unsupported operand type(s) for /: 'int' and 'NoneType'